# 生成テキストの多様性と繰り返しを測る (distinct-n)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_evaluation/distinct_n.ipynb)

NLP クックブック 17-15 ／ 評価指標と統計的検定

対話システムや生成モデルの出力が、単調で似た文ばかりになっていないか、同じ語句を繰り返していないかを数値化したい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
from collections import Counter
from fugashi import Tagger

tagger = Tagger("-Owakati")
tokenize = lambda s: tagger.parse(s).split()

def ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))

def distinct_n(texts, n):
    """出力全体での、異なり n-gram 数 ÷ 延べ n-gram 数"""
    all_ngrams = [g for t in texts for g in ngrams(tokenize(t), n)]
    return len(set(all_ngrams)) / max(1, len(all_ngrams))

def repetition_rate(text, n=3):
    """1 つの出力の中で、繰り返し現れた n-gram の割合"""
    counts = Counter(ngrams(tokenize(text), n))
    total = sum(counts.values())
    return sum(c for c in counts.values() if c > 1) / max(1, total)

bland = ["そうですね。", "それはいいですね。", "そうですね、いいと思います。", "わかりません。", "そうですね。"]
diverse = ["京都なら秋の東福寺がおすすめです。", "私は辛い料理が苦手なんです。", "最近は古い映画ばかり観ています。", "猫を二匹飼っています。", "明日は雨らしいので傘を忘れずに。"]
for name, texts in [("無難な応答", bland), ("多様な応答", diverse)]:
    print(f"{name}: distinct-1 = {distinct_n(texts, 1):.3f}  distinct-2 = {distinct_n(texts, 2):.3f}")

loop = "私は猫が好きです。私は猫が好きです。私は猫が好きです。なぜなら猫はかわいいからです。"
print("繰り返し率:", round(repetition_rate(loop), 3), "/", round(repetition_rate(diverse[0]), 3))

無難な応答: distinct-1 = 0.519  distinct-2 = 0.682
多様な応答: distinct-1 = 0.750  distinct-2 = 0.915
繰り返し率: 0.704 / 0.0


- distinct-n は出力の総量が増えると下がるので、比較するシステム間で出力数と長さを揃える。
- 多様性だけ高くても意味がない (ランダムな文字列が最も多様)。品質の指標と必ず組み合わせて見る。
- 複数の出力同士の類似度を測る Self-BLEU も、多様性の指標としてよく使われる (低いほど多様)。